In [9]:
import pandas as pd
import glob
import os

# 1. Get file list
path = 'path_to_csv_files'
all_files = glob.glob(os.path.join("HUPA*.csv"))

# 2. & 3. Read, add columns, and append to list
li = []
for filename in all_files:
    df = pd.read_csv(filename,sep=";")
    patientid = filename.split(".")[0]
    df['Patient_ID'] = os.path.basename(patientid) # Adding column
    li.append(df)

# Concatenate all into one DataFrame
df_final = pd.concat(li, ignore_index=True)

In [10]:
df_final

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,Patient_ID
0,2018-06-13T18:40:00,332.000000,6.35950,82.322835,34.0,0.091667,0.0,0.0,HUPA0001P
1,2018-06-13T18:45:00,326.000000,7.72800,83.740157,0.0,0.091667,0.0,0.0,HUPA0001P
2,2018-06-13T18:50:00,330.000000,4.74950,80.525180,0.0,0.091667,0.0,0.0,HUPA0001P
3,2018-06-13T18:55:00,324.000000,6.35950,89.129032,20.0,0.091667,0.0,0.0,HUPA0001P
4,2018-06-13T19:00:00,306.000000,5.15200,92.495652,0.0,0.075000,0.0,0.0,HUPA0001P
...,...,...,...,...,...,...,...,...,...
309387,2022-05-18T11:55:00,109.333333,10.79280,104.171171,0.0,0.000000,0.0,0.0,HUPA0028P
309388,2022-05-18T12:00:00,114.000000,9.80346,103.442623,0.0,0.000000,0.0,0.0,HUPA0028P
309389,2022-05-18T12:05:00,118.666667,5.66622,95.542857,0.0,0.000000,0.0,0.0,HUPA0028P
309390,2022-05-18T12:10:00,123.333333,5.57628,91.381356,0.0,0.000000,0.0,0.0,HUPA0028P


### Handling negative `bolus_volume_delivered`

##### **Reason:** `bolus_volume_delivered` contains values of -1 and -3 in HUPA0017P. Insulin delivery volume is a physical quantity and cannot be negative. These values could be incorrect reading from device. We replaced all negative bolus values with 0 which means that no insulin was delivered. If left uncleaned, negative bolus values would lower the average bolus dose and hinder any analysis of insulin-glucose response.

In [5]:
# Reading file
df_all = pd.read_excel("cleaned_data.xlsx")

# Checking negative bolus values per patient
negative_bolus = df_all[df_all['bolus_volume_delivered'] < 0]

print(f"Total negative bolus readings: {len(negative_bolus)}")

print(negative_bolus.groupby('patient_id')['bolus_volume_delivered'].value_counts())

# Cleaning bolus values
def clean_bolus(group):
    patient_id = group.name
    
    # Count bad readings
    bad_bolus = group['bolus_volume_delivered'] < 0
    
    print(f"{patient_id}: {bad_bolus.sum()} negative bolus readings found")
    
    # Replace negative values with 0
    group.loc[bad_bolus, 'bolus_volume_delivered'] = 0
    
    return group

# Apply cleaning
cols = df_all.columns

df_all = (df_all.groupby('patient_id', group_keys=False)[cols].apply(clean_bolus).reset_index(drop=True))

print("Bolus cleaning complete")

print(
    f"Remaining negative values: "
    f"{(df_all['bolus_volume_delivered'] < 0).sum()}"
)

Total negative bolus readings: 4
patient_id  bolus_volume_delivered
HUPA0017P   -1.0                      3
            -3.0                      1
Name: count, dtype: int64
HUPA0001P: 0 negative bolus readings found
HUPA0002P: 0 negative bolus readings found
HUPA0003P: 0 negative bolus readings found
HUPA0004P: 0 negative bolus readings found
HUPA0005P: 0 negative bolus readings found
HUPA0006P: 0 negative bolus readings found
HUPA0007P: 0 negative bolus readings found
HUPA0009P: 0 negative bolus readings found
HUPA0010P: 0 negative bolus readings found
HUPA0011P: 0 negative bolus readings found
HUPA0014P: 0 negative bolus readings found
HUPA0015P: 0 negative bolus readings found
HUPA0016P: 0 negative bolus readings found
HUPA0017P: 4 negative bolus readings found
HUPA0018P: 0 negative bolus readings found
HUPA0019P: 0 negative bolus readings found
HUPA0020P: 0 negative bolus readings found
HUPA0021P: 0 negative bolus readings found
HUPA0022P: 0 negative bolus readings found
HUPA0023P

### Rounding columns `glucose`, `calories` and `heart_rate` to 2 decimal places and `basal_rate` to 3 decimal places

##### **Reason:** Rounding columns `glucose`, `calories`, `heart_rate` and `basal_rate` for the ease of readability for insights and visualizations.

In [6]:
# Rounding the columns to 2 decimal places
df_all = df_all.round({'glucose': 2, 'calories': 2, 'heart_rate': 2, 'basal_rate': 3})

print("Change complete")

Change complete


In [8]:
df_all.to_excel("Team6_DataDynamos_Python-Hackathon_MAY2026_V2.xlsx", index=False, sheet_name='DataSheet')

print("New clean data file that contains handling of bolus_volume_delivered negative values + rounding some columns to 2 and 3 decimal places")

New clean data file that contains handling of bolus_volume_delivered negative values + rounding some columns to 2 and 3 decimal places
